# Centroid vs. Gaussian Drift Analysis

This notebook analyzes the relationship between centroid-based cosine drift and Gaussian Wasserstein-2 drift.

It covers:

- data loading and validation,
- modality-level descriptive statistics,
- Pearson and Spearman correlations,
- temporal drift trends,
- mean and spread components of Wasserstein drift,
- agreement and disagreement cases,
- export of result tables.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

try:
    from helper_functions import PROCESSED_DIR
except ImportError:
    PROJECT_ROOT = Path.cwd()
    PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

OUTPUT_PATH = PROCESSED_DIR / "07_centroid_gaussian_comparison"
FIGURE_PATH = OUTPUT_PATH / "figures"
TABLE_PATH = OUTPUT_PATH / "tables"

FIGURE_PATH.mkdir(parents=True, exist_ok=True)
TABLE_PATH.mkdir(parents=True, exist_ok=True)

COMBINED_PATH = OUTPUT_PATH / "centroid_gaussian_drift_comparison.parquet"
SUMMARY_PATH = OUTPUT_PATH / "modality_drift_summary.csv"

print("Processed directory:", PROCESSED_DIR)
print("Combined dataset:", COMBINED_PATH)
print("Summary dataset:", SUMMARY_PATH)


## 1. Load the results


In [ ]:
dataset_combined = pd.read_parquet(COMBINED_PATH)
modality_summary_saved = pd.read_csv(SUMMARY_PATH)

print("Combined shape:", dataset_combined.shape)
print("Saved summary shape:", modality_summary_saved.shape)

display(dataset_combined.head())
display(modality_summary_saved)


## 2. Inspect columns and data types


In [ ]:
display(
    pd.DataFrame({
        "column": dataset_combined.columns,
        "dtype": dataset_combined.dtypes.astype(str).values,
        "missing": dataset_combined.isna().sum().values,
        "missing_share": dataset_combined.isna().mean().values,
    })
)


## 3. Validate the merged comparison dataset


In [ ]:
merge_keys = [
    "modality",
    "genre",
    "window_start",
    "next_window_start",
]

required_columns = [
    *merge_keys,
    "cosine_distance",
    "w2_distance",
    "mean_component",
    "std_component",
]

missing_required = [
    column for column in required_columns
    if column not in dataset_combined.columns
]

if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

duplicate_count = dataset_combined.duplicated(merge_keys).sum()

numeric_columns = [
    "cosine_distance",
    "w2_distance",
    "mean_component",
    "std_component",
]

infinite_counts = {
    column: np.isinf(dataset_combined[column]).sum()
    for column in numeric_columns
}

validation = pd.DataFrame({
    "check": [
        "Duplicate transition rows",
        "Missing cosine distances",
        "Missing W2 distances",
        "Infinite cosine distances",
        "Infinite W2 distances",
        "Negative W2 distances",
        "Negative mean components",
        "Negative std components",
    ],
    "count": [
        duplicate_count,
        dataset_combined["cosine_distance"].isna().sum(),
        dataset_combined["w2_distance"].isna().sum(),
        infinite_counts["cosine_distance"],
        infinite_counts["w2_distance"],
        (dataset_combined["w2_distance"] < 0).sum(),
        (dataset_combined["mean_component"] < 0).sum(),
        (dataset_combined["std_component"] < 0).sum(),
    ],
})

display(validation)


In [ ]:
analysis_data = (
    dataset_combined
    .replace([np.inf, -np.inf], np.nan)
    .dropna(
        subset=[
            "cosine_distance",
            "w2_distance",
            "mean_component",
            "std_component",
        ]
    )
    .copy()
)

print("Rows available for analysis:", len(analysis_data))
print("Modalities:", sorted(analysis_data["modality"].unique()))
print("Genres:", analysis_data["genre"].nunique())


## 4. Descriptive statistics by modality

Cosine distance and Wasserstein distance are not directly comparable in numerical magnitude. Their summaries should be interpreted separately.


In [ ]:
descriptive_summary = (
    analysis_data
    .groupby("modality")
    .agg(
        n_valid_transitions=("cosine_distance", "size"),
        n_genres=("genre", "nunique"),
        mean_cosine_drift=("cosine_distance", "mean"),
        median_cosine_drift=("cosine_distance", "median"),
        std_cosine_drift=("cosine_distance", "std"),
        q25_cosine_drift=("cosine_distance", lambda x: x.quantile(0.25)),
        q75_cosine_drift=("cosine_distance", lambda x: x.quantile(0.75)),
        mean_w2_drift=("w2_distance", "mean"),
        median_w2_drift=("w2_distance", "median"),
        std_w2_drift=("w2_distance", "std"),
        q25_w2_drift=("w2_distance", lambda x: x.quantile(0.25)),
        q75_w2_drift=("w2_distance", lambda x: x.quantile(0.75)),
        mean_mean_component=("mean_component", "mean"),
        median_mean_component=("mean_component", "median"),
        mean_std_component=("std_component", "mean"),
        median_std_component=("std_component", "median"),
    )
    .reset_index()
)

display(descriptive_summary)


## 5. Correlations by modality

Spearman correlation is the primary measure because it evaluates whether the two metrics rank transitions similarly without assuming a linear relationship.

Pearson correlation is included as a secondary measure.


In [ ]:
correlation_rows = []

for modality, modality_data in analysis_data.groupby("modality"):
    cosine = modality_data["cosine_distance"]
    w2 = modality_data["w2_distance"]
    mean_component = modality_data["mean_component"]
    std_component = modality_data["std_component"]

    spearman_w2 = spearmanr(cosine, w2, nan_policy="omit")
    pearson_w2 = pearsonr(cosine, w2)
    spearman_mean = spearmanr(cosine, mean_component, nan_policy="omit")
    spearman_std = spearmanr(cosine, std_component, nan_policy="omit")

    correlation_rows.append({
        "modality": modality,
        "n": len(modality_data),
        "spearman_cosine_w2": spearman_w2.statistic,
        "spearman_cosine_w2_p": spearman_w2.pvalue,
        "pearson_cosine_w2": pearson_w2.statistic,
        "pearson_cosine_w2_p": pearson_w2.pvalue,
        "spearman_cosine_mean_component": spearman_mean.statistic,
        "spearman_cosine_mean_component_p": spearman_mean.pvalue,
        "spearman_cosine_std_component": spearman_std.statistic,
        "spearman_cosine_std_component_p": spearman_std.pvalue,
    })

correlation_summary = pd.DataFrame(correlation_rows)
display(correlation_summary)


In [ ]:
def correlation_strength(value):
    absolute = abs(value)

    if absolute < 0.10:
        return "negligible"
    if absolute < 0.30:
        return "weak"
    if absolute < 0.50:
        return "moderate"
    if absolute < 0.70:
        return "strong"
    return "very strong"

interpretation_table = correlation_summary[
    [
        "modality",
        "spearman_cosine_w2",
        "spearman_cosine_mean_component",
        "spearman_cosine_std_component",
    ]
].copy()

for column in interpretation_table.columns[1:]:
    interpretation_table[f"{column}_strength"] = (
        interpretation_table[column].apply(correlation_strength)
    )

display(interpretation_table)


### Interpretation guide

- High positive cosine–W2 correlation: both approaches tend to identify the same transitions as relatively dynamic.
- High cosine–mean-component correlation: centroid drift mainly agrees with movement of the Gaussian mean.
- Low cosine–std-component correlation: spread changes contain information that centroid drift does not capture.
- High cosine–std-component correlation: directional centroid changes often occur together with changes in internal spread.


## 6. Drift distributions by modality


In [ ]:
modalities = sorted(analysis_data["modality"].unique())

fig, ax = plt.subplots(figsize=(10, 6))
data = [
    analysis_data.loc[
        analysis_data["modality"] == modality,
        "cosine_distance"
    ].dropna()
    for modality in modalities
]
ax.boxplot(data, tick_labels=modalities, showfliers=False)
ax.set_title("Centroid cosine drift by modality")
ax.set_xlabel("Modality")
ax.set_ylabel("Cosine distance")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(FIGURE_PATH / "cosine_drift_by_modality.png", dpi=300)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
data = [
    analysis_data.loc[
        analysis_data["modality"] == modality,
        "w2_distance"
    ].dropna()
    for modality in modalities
]
ax.boxplot(data, tick_labels=modalities, showfliers=False)
ax.set_title("Gaussian Wasserstein-2 drift by modality")
ax.set_xlabel("Modality")
ax.set_ylabel("W2 distance")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(FIGURE_PATH / "w2_drift_by_modality.png", dpi=300)
plt.show()


## 7. Cosine drift versus W2 drift


In [ ]:
for modality, modality_data in analysis_data.groupby("modality"):
    rho = spearmanr(
        modality_data["cosine_distance"],
        modality_data["w2_distance"],
        nan_policy="omit",
    ).statistic

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(
        modality_data["cosine_distance"],
        modality_data["w2_distance"],
        alpha=0.45,
        s=18,
    )
    ax.set_title(f"{modality}: cosine drift vs. W2 drift, Spearman ρ={rho:.3f}")
    ax.set_xlabel("Cosine distance")
    ax.set_ylabel("W2 distance")
    fig.tight_layout()
    fig.savefig(
        FIGURE_PATH / f"{modality}_cosine_vs_w2.png",
        dpi=300,
    )
    plt.show()


## 8. Mean movement versus spread change


In [ ]:
for modality, modality_data in analysis_data.groupby("modality"):
    fig, ax = plt.subplots(figsize=(7, 5))
    scatter = ax.scatter(
        modality_data["mean_component"],
        modality_data["std_component"],
        c=modality_data["cosine_distance"],
        alpha=0.5,
        s=20,
    )
    ax.set_title(f"{modality}: Gaussian mean and spread components")
    ax.set_xlabel("Mean component")
    ax.set_ylabel("Standard-deviation component")
    fig.colorbar(scatter, ax=ax, label="Cosine distance")
    fig.tight_layout()
    fig.savefig(
        FIGURE_PATH / f"{modality}_mean_vs_std_component.png",
        dpi=300,
    )
    plt.show()


## 9. Relative contribution of mean and spread

The squared W2 distance is the sum of the squared mean and standard-deviation components in the diagonal Gaussian implementation.


In [ ]:
analysis_data["mean_share_w2_squared"] = np.where(
    analysis_data["w2_distance"] > 0,
    analysis_data["mean_component"] ** 2
    / analysis_data["w2_distance"] ** 2,
    np.nan,
)

analysis_data["std_share_w2_squared"] = np.where(
    analysis_data["w2_distance"] > 0,
    analysis_data["std_component"] ** 2
    / analysis_data["w2_distance"] ** 2,
    np.nan,
)

component_summary = (
    analysis_data
    .groupby("modality")
    .agg(
        mean_mean_share=("mean_share_w2_squared", "mean"),
        median_mean_share=("mean_share_w2_squared", "median"),
        mean_std_share=("std_share_w2_squared", "mean"),
        median_std_share=("std_share_w2_squared", "median"),
    )
    .reset_index()
)

display(component_summary)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
component_plot = component_summary.set_index("modality")[
    ["mean_mean_share", "mean_std_share"]
]
component_plot.plot(kind="bar", stacked=True, ax=ax)
ax.set_title("Average contribution to squared W2 drift")
ax.set_xlabel("Modality")
ax.set_ylabel("Average share")
ax.legend(["Mean movement", "Spread change"])
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(FIGURE_PATH / "w2_component_shares.png", dpi=300)
plt.show()


## 10. Temporal drift trends

The median is used because drift distributions may contain strong outliers.


In [ ]:
temporal_summary = (
    analysis_data
    .groupby(["modality", "next_window_start"])
    .agg(
        n_transitions=("genre", "size"),
        n_genres=("genre", "nunique"),
        median_cosine_drift=("cosine_distance", "median"),
        mean_cosine_drift=("cosine_distance", "mean"),
        median_w2_drift=("w2_distance", "median"),
        mean_w2_drift=("w2_distance", "mean"),
    )
    .reset_index()
)

display(temporal_summary.head())


In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

for modality, modality_data in temporal_summary.groupby("modality"):
    ax.plot(
        modality_data["next_window_start"],
        modality_data["median_cosine_drift"],
        label=modality,
    )

ax.set_title("Median centroid drift over time")
ax.set_xlabel("Next window start")
ax.set_ylabel("Median cosine distance")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_PATH / "temporal_median_cosine_drift.png", dpi=300)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

for modality, modality_data in temporal_summary.groupby("modality"):
    ax.plot(
        modality_data["next_window_start"],
        modality_data["median_w2_drift"],
        label=modality,
    )

ax.set_title("Median Gaussian W2 drift over time")
ax.set_xlabel("Next window start")
ax.set_ylabel("Median W2 distance")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_PATH / "temporal_median_w2_drift.png", dpi=300)
plt.show()


## 11. Agreement and disagreement cases

The metrics are converted to within-modality percentile ranks. This avoids comparing their raw numerical scales directly.


In [ ]:
analysis_data["cosine_percentile"] = (
    analysis_data
    .groupby("modality")["cosine_distance"]
    .rank(pct=True)
)

analysis_data["w2_percentile"] = (
    analysis_data
    .groupby("modality")["w2_distance"]
    .rank(pct=True)
)

analysis_data["rank_difference"] = (
    analysis_data["w2_percentile"]
    - analysis_data["cosine_percentile"]
)

analysis_data["absolute_rank_difference"] = (
    analysis_data["rank_difference"].abs()
)

analysis_data["agreement_group"] = np.select(
    [
        (analysis_data["cosine_percentile"] >= 0.90)
        & (analysis_data["w2_percentile"] >= 0.90),
        (analysis_data["cosine_percentile"] <= 0.10)
        & (analysis_data["w2_percentile"] <= 0.10),
        (analysis_data["cosine_percentile"] <= 0.25)
        & (analysis_data["w2_percentile"] >= 0.75),
        (analysis_data["cosine_percentile"] >= 0.75)
        & (analysis_data["w2_percentile"] <= 0.25),
    ],
    [
        "high_both",
        "low_both",
        "high_w2_low_cosine",
        "high_cosine_low_w2",
    ],
    default="other",
)

agreement_counts = (
    analysis_data
    .groupby(["modality", "agreement_group"])
    .size()
    .reset_index(name="n")
)

display(agreement_counts)


In [ ]:
example_columns = [
    "modality",
    "genre",
    "window_start",
    "next_window_start",
    "cosine_distance",
    "w2_distance",
    "mean_component",
    "std_component",
    "cosine_percentile",
    "w2_percentile",
    "rank_difference",
    "agreement_group",
]

agreement_examples = (
    analysis_data.loc[
        analysis_data["agreement_group"] != "other",
        example_columns,
    ]
    .sort_values(
        ["agreement_group", "absolute_rank_difference"]
        if "absolute_rank_difference" in example_columns
        else ["agreement_group", "rank_difference"],
        ascending=[True, False],
    )
)

display(agreement_examples.head(30))


In [ ]:
largest_disagreements = (
    analysis_data[
        [
            *example_columns,
            "absolute_rank_difference",
            "mean_share_w2_squared",
            "std_share_w2_squared",
        ]
    ]
    .sort_values("absolute_rank_difference", ascending=False)
    .head(50)
)

display(largest_disagreements)


## 12. Genres with the highest average drift


In [ ]:
genre_summary = (
    analysis_data
    .groupby(["modality", "genre"])
    .agg(
        n_transitions=("cosine_distance", "size"),
        mean_cosine_drift=("cosine_distance", "mean"),
        median_cosine_drift=("cosine_distance", "median"),
        mean_w2_drift=("w2_distance", "mean"),
        median_w2_drift=("w2_distance", "median"),
        mean_std_component=("std_component", "mean"),
    )
    .reset_index()
)

genre_summary["cosine_rank_within_modality"] = (
    genre_summary
    .groupby("modality")["mean_cosine_drift"]
    .rank(ascending=False, method="min")
)

genre_summary["w2_rank_within_modality"] = (
    genre_summary
    .groupby("modality")["mean_w2_drift"]
    .rank(ascending=False, method="min")
)

top_genres_cosine = (
    genre_summary
    .sort_values(
        ["modality", "mean_cosine_drift"],
        ascending=[True, False],
    )
    .groupby("modality")
    .head(10)
)

top_genres_w2 = (
    genre_summary
    .sort_values(
        ["modality", "mean_w2_drift"],
        ascending=[True, False],
    )
    .groupby("modality")
    .head(10)
)

display(top_genres_cosine)
display(top_genres_w2)


## 13. Export analysis tables


In [ ]:
descriptive_summary.to_csv(
    TABLE_PATH / "descriptive_summary_by_modality.csv",
    index=False,
)

correlation_summary.to_csv(
    TABLE_PATH / "correlations_by_modality.csv",
    index=False,
)

component_summary.to_csv(
    TABLE_PATH / "w2_component_summary_by_modality.csv",
    index=False,
)

temporal_summary.to_csv(
    TABLE_PATH / "temporal_drift_summary.csv",
    index=False,
)

agreement_counts.to_csv(
    TABLE_PATH / "agreement_group_counts.csv",
    index=False,
)

largest_disagreements.to_csv(
    TABLE_PATH / "largest_drift_disagreements.csv",
    index=False,
)

genre_summary.to_csv(
    TABLE_PATH / "genre_drift_summary.csv",
    index=False,
)

analysis_data.to_parquet(
    OUTPUT_PATH / "centroid_gaussian_drift_analysis_enriched.parquet",
    index=False,
)

print("Tables saved to:", TABLE_PATH)
print("Figures saved to:", FIGURE_PATH)


## 14. Thesis-oriented result prompts

Use the generated tables to answer:

1. Which modalities show the strongest and weakest median centroid drift?
2. Which modalities show the strongest and weakest median Gaussian W2 drift?
3. Do centroid and Gaussian drift rank transitions similarly?
4. Is the relationship mainly explained by movement of the Gaussian mean?
5. Which modalities show substantial spread changes that cosine drift does not capture?
6. Which genres and temporal transitions produce the strongest agreement?
7. Which cases challenge the centroid-based conclusion?
8. Do temporal peaks coincide across the two representations?
9. Could major peaks be explained by low numbers of valid genres or transitions?
10. Are the conclusions stable across modalities or modality-specific?
